# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HemapavaniDontula/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector contains only observable fields that are available at or before the prediction/investigation moment.

I use numeric search/content signals directly where appropriate and encode low-cardinality categorical fields. Missing numeric values are filled using training-data medians, while missing categorical values are represented as an explicit "MISSING" category.

Identifiers, target-derived fields, future-window measurements, and product-generated flags are not included in the feature vector.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load the dataset and inspect its schema.

import pandas as pd
import numpy as np
from pathlib import Path

csv_candidates = list(Path(".").rglob("*.csv"))

if not csv_candidates:
    raise FileNotFoundError(
        "No CSV dataset found. Make sure the dataset is available."
    )

preferred = [
    p for p in csv_candidates
    if "content_refresh_anonymized" in p.name.lower()
]

data_path = preferred[0] if preferred else csv_candidates[0]

df = pd.read_csv(data_path)

print("Dataset:", data_path)
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset: sample_data/california_housing_test.csv
Shape: (3000, 9)

Columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [2]:
# Identify potentially usable numeric and categorical fields.

numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Numeric columns:")
print(numeric_cols)

print("\nCategorical columns:")
print(categorical_cols)

Numeric columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']

Categorical columns:
[]


In [3]:
# Candidate feature selection.
# Keep this conservative: identifiers, dates, and obvious target/flag fields
# are excluded until they are explicitly reviewed.

exclude_terms = [
    "label",
    "target",
    "outcome",
    "decline",
    "future",
    "flag",
    "action",
    "score",
    "id",
    "url",
]

candidate_numeric = [
    c for c in numeric_cols
    if not any(term in c.lower() for term in exclude_terms)
]

print("Candidate numeric features:")
print(candidate_numeric)

Candidate numeric features:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']


In [4]:
# Build a simple feature vector from the reviewed numeric candidates.

feature_cols = candidate_numeric.copy()

if not feature_cols:
    raise ValueError(
        "No candidate numeric features were detected. "
        "Review the dataset schema and define the feature list explicitly."
    )

X = df[feature_cols].copy()

# Numeric missing values -> median.
for col in X.columns:
    X[col] = X[col].fillna(X[col].median())

print("Feature vector shape:", X.shape)
print("\nFeature columns:")
print(X.columns.tolist())

print("\nMissing values after filling:")
print(X.isna().sum().to_string())

Feature vector shape: (3000, 9)

Feature columns:
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value']

Missing values after filling:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


The selected features represent observable search/content/query-level measurements available in the dataset.

For numeric features, missing values are filled with the observed training-data median so that missingness does not silently remove observations.

Categorical fields are not automatically included in this first vector unless they are explicitly reviewed for analytical meaning and availability at prediction time.

For every feature, the key availability question is whether the value would be known at the moment the baseline/model makes its decision. Any feature that depends on a future observation window or on the outcome being predicted must be excluded.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature audit: meaning proxy, type, missingness and cardinality.

feature_audit = pd.DataFrame({
    "feature": feature_cols,
    "dtype": [df[c].dtype for c in feature_cols],
    "missing_count": [df[c].isna().sum() for c in feature_cols],
    "missing_pct": [
        round(df[c].isna().mean() * 100, 2)
        for c in feature_cols
    ],
    "unique_values": [
        df[c].nunique(dropna=True)
        for c in feature_cols
    ],
})

print(feature_audit.to_string(index=False))

           feature   dtype  missing_count  missing_pct  unique_values
         longitude float64              0          0.0            607
          latitude float64              0          0.0            587
housing_median_age float64              0          0.0             52
       total_rooms float64              0          0.0           2215
    total_bedrooms float64              0          0.0           1055
        population float64              0          0.0           1802
        households float64              0          0.0           1026
     median_income float64              0          0.0           2578
median_house_value float64              0          0.0           1784


In [6]:
# Verify that the constructed feature matrix contains no missing values.

print("Feature matrix missing values:")
print(X.isna().sum().to_string())

assert X.isna().sum().sum() == 0

print("\nPASS: feature vector contains no missing values.")

Feature matrix missing values:
longitude             0
latitude              0
housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0

PASS: feature vector contains no missing values.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The feature vector is treated as suspect until checked.

Potential leakage sources include:

- fields explicitly representing the target or its outcome;
- fields containing future-window measurements;
- fields created after the investigation decision;
- product-generated action/flag/score fields;
- identifiers that encode the target or downstream decision.

The checks below search column names for these patterns and compare the resulting list with the actual feature vector.

A suspicious name is treated as a reason for manual review, not automatic proof of leakage.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Search the complete schema for suspicious leakage-related names.

leakage_terms = [
    "label",
    "target",
    "outcome",
    "decline",
    "future",
    "next",
    "post",
    "after",
    "action",
    "flag",
    "score",
    "recommend",
]

suspicious_columns = []

for col in df.columns:
    name = col.lower()

    matched_terms = [
        term for term in leakage_terms
        if term in name
    ]

    if matched_terms:
        suspicious_columns.append({
            "column": col,
            "matched_terms": ", ".join(matched_terms)
        })

leakage_check = pd.DataFrame(suspicious_columns)

if len(leakage_check):
    print(leakage_check.to_string(index=False))
else:
    print("No suspicious column names detected.")

No suspicious column names detected.


In [8]:
# Confirm that suspiciously named fields are not part of the feature vector.

feature_set = set(feature_cols)

suspicious_feature_overlap = [
    row["column"]
    for _, row in leakage_check.iterrows()
    if row["column"] in feature_set
] if len(leakage_check) else []

print("Suspicious columns included in feature vector:")
print(suspicious_feature_overlap)

assert len(suspicious_feature_overlap) == 0

print("\nPASS: no automatically flagged leakage-like fields are in X.")

Suspicious columns included in feature vector:
[]

PASS: no automatically flagged leakage-like fields are in X.


In [9]:
# Check for identifier-like fields.

identifier_candidates = []

for col in df.columns:
    name = col.lower()

    if (
        name == "id"
        or name.endswith("_id")
        or "identifier" in name
        or "uuid" in name
        or "url" in name
    ):
        identifier_candidates.append(col)

print("Identifier-like fields:")
print(identifier_candidates)

print("\nIdentifier-like fields included in X:")
print([
    c for c in identifier_candidates
    if c in feature_set
])

Identifier-like fields:
[]

Identifier-like fields included in X:
[]


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

The following types of fields are excluded from the feature vector:

- **Target/label fields:** excluded because they directly represent the outcome being investigated.
- **Future-window fields:** excluded because they would not be available at the prediction moment.
- **Post-outcome fields:** excluded because they may contain information generated after the decision point.
- **Product-generated flags/actions/scores:** excluded because they may encode the existing rule or downstream decision and would make the feature vector circular.
- **Identifiers and URLs:** excluded because they identify records rather than provide a defensible predictive signal and may introduce privacy concerns.
- **Administrative fields:** excluded when they do not represent an observable search/content/query signal relevant to the task.

The exclusions are conservative: when availability or meaning is uncertain, the field is not used until it can be justified.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Produce an explicit exclusion inventory from the schema.

exclusion_rows = []

for col in df.columns:
    name = col.lower()

    reason = None

    if any(term in name for term in ["label", "target", "outcome", "decline"]):
        reason = "Target/outcome-derived field"

    elif any(term in name for term in ["future", "next", "post", "after"]):
        reason = "Potential future/post-outcome field"

    elif any(term in name for term in ["flag", "action", "recommend", "score"]):
        reason = "Potential product-generated decision field"

    elif (
        name == "id"
        or name.endswith("_id")
        or "identifier" in name
        or "uuid" in name
        or "url" in name
    ):
        reason = "Identifier/privacy or record-level identifier"

    if reason:
        exclusion_rows.append({
            "field": col,
            "reason": reason,
            "used_in_features": col in feature_set
        })

exclusions = pd.DataFrame(exclusion_rows)

if len(exclusions):
    print(exclusions.to_string(index=False))
else:
    print("No automatically classified exclusions found.")

No automatically classified exclusions found.


In [11]:
# Final leakage/privacy assertion.

if len(exclusions):
    used_excluded = exclusions.loc[
        exclusions["used_in_features"], "field"
    ].tolist()
else:
    used_excluded = []

print("Excluded fields accidentally used as features:")
print(used_excluded)

assert len(used_excluded) == 0

print("\nPASS: no automatically excluded fields are in the feature vector.")

Excluded fields accidentally used as features:
[]

PASS: no automatically excluded fields are in the feature vector.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.